# 020 Create Silver
Take the data from the Bronze table and tidy it up for Silver layer.  This involves:
- cleaning column names
- setting correct datatypes
- any other transforms


## Setup


### Parameters


In [1]:
debug_yn = True


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, 3, Finished, Available, Finished, False)

### Imports


In [2]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import DateType, DecimalType, FloatType, IntegerType, StructField, StructType


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, 4, Finished, Available, Finished, False)

### Reset


In [3]:
%%sql
DROP TABLE IF EXISTS hpi_silver;


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Main


### Get the Bronze table


In [4]:
df = spark.table("hpi_bronze")


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, 6, Finished, Available, Finished, False)

In [5]:
if debug_yn == True:
    display(df.limit(10))


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bfceef72-7a8f-40d4-bfc3-69fd784f11fc)

### Cast datatypes


In [6]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import IntegerType, StringType, DecimalType

# Define your column lists; ignore Strings as they are already string
date_columns = ["Date"]
int_columns = ["AveragePrice", "AveragePriceSA", "SalesVolume", "DetachedPrice", "SemiDetachedPrice", "TerracedPrice", "FlatPrice", "CashPrice", "CashSalesVolume", "MortgagePrice", "MortgageSalesVolume", "FTBPrice", "FOOPrice", "NewPrice", "NewSalesVolume", "OldPrice", "OldSalesVolume"]
decimal_columns = ["Index", "IndexSA", "1m%change", "12m%change", "DetachedIndex", "Detached1m%Change", "Detached12m%Change", "SemiDetachedIndex", "SemiDetached1m%Change", "SemiDetached12m%Change", "TerracedIndex", "Terraced1m%Change", "Terraced12m%Change", "FlatIndex", "Flat1m%Change", "Flat12m%Change", "CashIndex", "Cash1m%Change", "Cash12m%Change", "MortgageIndex", "Mortgage1m%Change", "Mortgage12m%Change", "FTBIndex", "FTB1m%Change", "FTB12m%Change", "FOOIndex", "FOO1m%Change", "FOO12m%Change", "NewIndex", "New1m%Change", "New12m%Change", "OldIndex", "Old1m%Change", "Old12m%Change"]

# Optional: custom format for date strings
date_format = "dd/MM/yyyy"

silver_df = df

# Cast date columns with formatting
for c in date_columns:
    silver_df = silver_df.withColumn(c, to_date(col(c), date_format))

# Cast integer columns
for c in int_columns:
    silver_df = silver_df.withColumn(c, col(c).cast(IntegerType()))

# Cast decimal columns with fixed precision/scale
for c in decimal_columns:
    silver_df = silver_df.withColumn(c, col(c).cast(DecimalType(10, 2)))


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, 8, Finished, Available, Finished, False)

In [7]:
if debug_yn == True:
    display(silver_df.limit(10))


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 51093149-10a0-4254-860c-fe472c01507e)

## Write to Silver


In [8]:
silver_df.write.mode("overwrite").format("delta").saveAsTable("hpi_silver")


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, 10, Finished, Available, Finished, False)

### Examples


In [9]:
# Transform the data ready for insert into the fact
from pyspark.sql.functions import concat, initcap, lit, regexp_replace, translate

# Get the table
df = spark.table("raw_data")

# Add cleaned columns:
# User name; capitalise and remove underscores
# Building name; capitalise and remove underscores
# Printer name; capitalise and remove underscores

df = df.withColumn("userLogin", initcap(translate("login", "_", " ")))
df = df.withColumn("tieredLogin", initcap(concat(translate("login", "_", " "), lit(" Tier 1"))))
df = df.withColumn("buildingName", initcap(translate("building", "_", " ")))
df = df.withColumn("printerName", initcap(translate("printer", "_", " ")))

## !!TODO don't need to do both
#df.createOrReplaceTempView("cleanData")
df.write.mode("overwrite").format("delta").saveAsTable("silver_data")

display(df.limit(10))


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, 11, Finished, Available, Finished, False)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `raw_data` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'UnresolvedRelation [raw_data], [], false


### More Pythonic version


In [ ]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import IntegerType, DecimalType

# Column: target_type or (target_type, format/precision)
column_casts = {
    "Date": ("date", "dd/MM/yyyy"),
    "AveragePrice": "int",
    "AveragePriceSA": "int",
    "SalesVolume": "int",
    "DetachedPrice": "int",
    "SemiDetachedPrice": "int",
    "TerracedPrice": "int",
    "FlatPrice": "int",
    "CashPrice": "int",
    "CashSalesVolume": "int",
    "MortgagePrice": "int",
    "MortgageSalesVolume": "int",
    "FTBPrice": "int",
    "FOOPrice": "int",
    "NewPrice": "int",
    "NewSalesVolume": "int",
    "OldPrice": "int",
    "OldSalesVolume": "int",
    **{name: "decimal" for name in [
        "Index", "IndexSA", "1m%change", "12m%change",
        "DetachedIndex", "Detached1m%Change", "Detached12m%Change",
        "SemiDetachedIndex", "SemiDetached1m%Change", "SemiDetached12m%Change",
        "TerracedIndex", "Terraced1m%Change", "Terraced12m%Change",
        "FlatIndex", "Flat1m%Change", "Flat12m%Change",
        "CashIndex", "Cash1m%Change", "Cash12m%Change",
        "MortgageIndex", "Mortgage1m%Change", "Mortgage12m%Change",
        "FTBIndex", "FTB1m%Change", "FTB12m%Change",
        "FOOIndex", "FOO1m%Change", "FOO12m%Change",
        "NewIndex", "New1m%Change", "New12m%Change",
        "OldIndex", "Old1m%Change", "Old12m%Change"
    ]}
}

# Define how to cast each type
type_handlers = {
    "int": lambda c, *args: col(c).cast(IntegerType()),
    "decimal": lambda c, *args: col(c).cast(DecimalType(10, 2)),
    "date": lambda c, fmt: to_date(col(c), fmt),
}

# Apply all casts in one loop
silver_df = df
for col_name, type_info in column_casts.items():
    type_str, *params = type_info if isinstance(type_info, tuple) else (type_info,)
    silver_df = silver_df.withColumn(col_name, type_handlers[type_str](col_name, *params))


display(silver_df)


StatementMeta(, 30850ff2-9cc6-4c77-931a-20af0f490efc, -1, Cancelled, , Cancelled, True)